# Search Capabilities — No Server Required

Spicy Regs publishes all data as public files on Cloudflare R2. That means you can do mirrulations-search-style queries (and more) from any Python shell without running a backend.

This notebook walks through the three layers of search available today:

| Layer | Use when | Scope | Latency |
|---|---|---|---|
| 1. Prebuilt docket index (`docket_search.json.gz`) | You want the same fast metadata search the site uses | Docket title + abstract (346K) | One ~10 MB download, then in-memory |
| 2. DuckDB over `dockets.parquet` | You need SQL flexibility on docket metadata | Docket title + abstract + fields | <1s per query |
| 3. DuckDB over partitioned `comments/` | Full-text over comment bodies (the mirrulations-search equivalent) | 24M+ comments | Seconds if you narrow by agency/docket |

Everything below hits public URLs. No auth, no local downloads of multi-GB files.

## Setup

In [9]:
import duckdb

# Custom R2 domain — avoids the rate limits on the pub-*.r2.dev dev endpoint.
R2_BASE_URL = "https://r2.spicy-regs.dev"

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
print("ready — all queries will stream directly from R2")

ready — all queries will stream directly from R2


## Layer 1 — Prebuilt docket search index

This is the exact blob the `/search` page on the live site loads into MiniSearch. It's a gzipped JSON with every docket's `id`, `agency_code`, `title`, `docket_type`, `modify_date`, and `abstract` (fields abbreviated as `id/a/t/x/d/s` for payload size — see `src/spicy_regs/pipeline/build_search_index.py`).

**When to reach for this:** you want docket-level fuzzy/prefix search with zero warm-up cost per query. Load once, then filter in pure Python.

The site uses [`minisearch`](https://github.com/lucaong/minisearch) for ranked fuzzy/prefix matching — the cell below keeps it simple with plain substring filtering so there's nothing to install. For production-quality ranking, feed these docs into your text-search lib of choice (Whoosh, Tantivy, even BM25 via `rank_bm25`).

In [10]:
import gzip
import json
import time
import urllib.error
import urllib.request
from pathlib import Path

INDEX_URL = f"{R2_BASE_URL}/docket_search.json.gz"
CACHE_PATH = Path(".cache/docket_search.json.gz")

# r2.dev public buckets rate-limit dev traffic (429). Cache locally so
# re-running this notebook doesn't re-hit R2, and retry with backoff on
# the first fetch. Delete .cache/docket_search.json.gz to force a refresh.
def _fetch_with_retry(url: str, attempts: int = 4) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "spicy-regs-notebook"})
    for i in range(attempts):
        try:
            with urllib.request.urlopen(req) as resp:
                return resp.read()
        except urllib.error.HTTPError as e:
            if e.code != 429 or i == attempts - 1:
                raise
            wait = 2 ** i
            print(f"  429 rate-limited, retrying in {wait}s…")
            time.sleep(wait)
    raise RuntimeError("unreachable")

if CACHE_PATH.exists():
    raw = CACHE_PATH.read_bytes()
    print(f"using cached index at {CACHE_PATH}")
else:
    print(f"fetching {INDEX_URL}…")
    raw = _fetch_with_retry(INDEX_URL)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_bytes(raw)

# R2 may or may not auto-decompress. Detect the gzip magic bytes (1f 8b).
if raw[:2] == b"\x1f\x8b":
    raw = gzip.decompress(raw)
payload = json.loads(raw)

print(f"index version: {payload['version']}")
print(f"dockets indexed: {payload['count']:,}")
payload['docs'][0]

fetching https://r2.spicy-regs.dev/docket_search.json.gz…
index version: 2026-04-14T21:14:37Z
dockets indexed: 273,145


{'id': 'WCPO-2026-0232',
 'a': 'WCPO',
 't': "Miner's Claim for Benefits Under the BLBA and Employment History",
 'x': 'Nonrulemaking',
 'd': '2026-04-14T15:23:34Z',
 's': "CM-911 is the standard application form filed by the miner for benefits under the Black Lung Benefits Act. The applicant lists the coal miner's work history on the CM-911a, and this form is completed by all applicants, both miners and survivors."}

In [18]:
def search_docket_index(query: str, agency: str | None = None, limit: int = 10):
    """Plain substring search over the prebuilt index.

    Fields: id, a=agency, t=title, x=docket_type, d=modify_date, s=abstract.
    Swap this for minisearch/whoosh/BM25 if you need ranking.
    """
    q = query.lower()
    hits = []
    for doc in payload['docs']:
        if agency and doc.get('a') != agency:
            continue
        if q in doc.get('t', '').lower() or q in doc.get('s', '').lower():
            hits.append(doc)
            if len(hits) >= limit:
                break
    return hits

for hit in search_docket_index("biotech", limit=5):
    print(f"[{hit['a']}] {hit['id']} — {hit['t']}")

[EPA] EPA-HQ-OPP-2016-0260 — Notice of Receipt - New Active Ingredient 1-Triacontanol for Registration as EPA Reg. No. 90866-EN
[FDA] FDA-2022-D-2512 — Q5A(R2) Viral Safety Evaluation of Biotechnology Products Derived from Cell Lines of Human or Animal Origin; International Council for Harmonisation; Draft Guidance for Industry; Availability
[BIS] BIS-2024-0050 — Controls on Certain Laboratory Equipment and Related Technology to Address Dual Use Concerns about Biotechnology
[EPA] EPA-HQ-OPP-2024-0518 — Mesotrione- Pesticide Petition (2F9014)- Establish Tolerances for Mesotrione-resistant Soybean
[BIS] BIS-2024-0054 — Impact of the Implementation of the Chemical Weapons Convention (CWC) on Legitimate Commercial Chemical, Biotechnology, and Pharmaceutical Activities Involving “Schedule 1” Chemicals (including “Schedule 1” Chemicals produced as Intermediates) During Calendar Year 2024


## Layer 2 — DuckDB over `dockets.parquet`

When you need SQL. 346K rows, so `ILIKE` across title + abstract is plenty fast — DuckDB streams the Parquet file from R2 and pushes down the predicate.

**When to reach for this:** you want to combine keyword matching with richer filters (agency, date windows, docket type) or join results into a pandas DataFrame.

In [12]:
conn.execute(f"""
    SELECT docket_id, agency_code, docket_type, modify_date, title
    FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
    WHERE (title ILIKE '%PFAS%' OR abstract ILIKE '%PFAS%')
      AND agency_code = 'EPA'
      AND modify_date >= '2024-01-01'
    ORDER BY modify_date DESC
    LIMIT 10
""").fetchdf()

,docket_id,agency_code,docket_type,modify_date,title
0,EPA-HQ-OPPT-2020-0549,EPA,Rulemaking,2026-01-14T18:23:07Z,Reporting and Recordkeeping for Perfluoroalkyl...
1,EPA-HQ-TRI-2021-0049,EPA,Rulemaking,2025-12-29T20:27:42Z,NDAA Mandated Addition of Certain Per- and Pol...
2,EPA-HQ-TRI-2022-0270,EPA,Rulemaking,2025-12-29T20:27:17Z,Changes to Reporting Requirements for Per- and...
3,EPA-HQ-TRI-2022-0453,EPA,Rulemaking,2025-12-29T20:27:04Z,Implementing Statutory Addition of Certain Per...
4,EPA-HQ-OPPT-2023-0538,EPA,Rulemaking,2025-10-08T19:07:25Z,Addition of Certain Per- and Polyfluoroalkyl S...
5,EPA-HQ-OPPT-2023-0499,EPA,Nonrulemaking,2025-06-17T15:23:46Z,TSCA Section 4 Test Orders for the PFAS Nation...
6,EPA-HQ-OW-2024-0454,EPA,Nonrulemaking,2025-05-14T14:08:30Z,National Recommended Ambient Water Quality Cri...
7,EPA-HQ-ORD-2019-0275,EPA,Nonrulemaking,2025-05-08T21:33:41Z,IRIS PFAS Toxicity Assessments Project (Genera...
8,EPA-HQ-OPPT-2024-0507,EPA,Rulemaking,2025-03-21T19:46:40Z,Clarification to the Toxics Release Inventory ...
9,EPA-HQ-OPPT-2020-0565,EPA,Nonrulemaking,2025-03-05T17:32:50Z,Response to TSCA Section 21 Petition: Coalitio...


## Layer 3 — Full-text over comments (partition-aware)

This is the mirrulations-search equivalent: search inside the 24M+ comment bodies.

### Anti-pattern: do not scan the flat file

There is a monolithic `comments.parquet` (~3 GB) meant for full-dataset analytics. **Don't use it for keyword search** — you'll burn bandwidth and minutes pulling data you throw away.

### Correct pattern: use `comments_index.parquet` to narrow

Comments on R2 are hive-partitioned at `comments/agency_code={A}/docket_id={D}/year={Y}/month={M}/part-0.parquet`. The tiny `comments_index.parquet` file tells you which partitions exist — query it first, then only read the matching parts.

This mirrors the production pattern in `frontend/src/lib/duckdb/useDuckDBService.ts` (`buildCommentsSource`).

In [13]:
# Peek the index — one row per (agency, docket, year, month) partition that exists.
conn.execute(f"""
    SELECT agency_code, docket_id, year, month, row_count
    FROM read_parquet('{R2_BASE_URL}/comments_index.parquet')
    WHERE agency_code = 'EPA'
    ORDER BY row_count DESC
    LIMIT 5
""").fetchdf()

,agency_code,docket_id,year,month,row_count
0,EPA,EPA-HQ-OA-2017-0190,2017,5,34249
1,EPA,EPA-HQ-OA-2017-0190,2017,6,24413
2,EPA,EPA-HQ-OAR-2025-0194,2025,10,16889
3,EPA,EPA-HQ-OW-2025-0322,2026,1,14800
4,EPA,EPA-HQ-OAR-2009-0517,2010,1,10338


In [16]:
MAX_PARTITIONS = 50  # keep fan-out under r2.dev's rate limit


def search_comments(
    phrase: str,
    agency: str,
    docket_id: str | None = None,
    year: int | None = None,
    limit: int = 20,
    max_partitions: int = MAX_PARTITIONS,
):
    """Full-text search over comment bodies, scoped to a single agency.

    Two-step query:
      1. Find matching partitions in comments_index.parquet.
      2. Scan only those partition files for the phrase.

    r2.dev rate-limits public buckets, and DuckDB issues one HTTP range
    request per partition file. Searching all of EPA's comments (hundreds
    of dockets) will 429. Narrow with `docket_id` or `year`, or raise
    `max_partitions` and accept the risk.
    """
    filters = [f"agency_code = '{agency}'"]
    if docket_id:
        filters.append(f"docket_id = '{docket_id}'")
    if year:
        filters.append(f"year = {year}")

    parts = conn.execute(f"""
        SELECT agency_code, docket_id, year, month, row_count
        FROM read_parquet('{R2_BASE_URL}/comments_index.parquet')
        WHERE {' AND '.join(filters)}
        ORDER BY row_count DESC
    """).fetchall()

    if not parts:
        return []


    urls = [
        f"'{R2_BASE_URL}/comments/agency_code={a}/docket_id={d}/year={y}/month={m}/part-0.parquet'"
        for (a, d, y, m, _rc) in parts
    ]

    print(f"scanning {len(urls)} partition(s)…")
    return conn.execute(f"""
        SELECT comment_id, docket_id, posted_date,
               substr(comment, 1, 240) AS preview
        FROM read_parquet([{', '.join(urls)}], union_by_name=true)
        WHERE comment ILIKE '%{phrase}%'
        ORDER BY posted_date DESC
        LIMIT {limit}
    """).fetchdf()


# Example: narrow to one docket — cheapest, always safe.
search_comments("methane", agency="EPA", docket_id="EPA-HQ-OAR-2021-0317", limit=5)

scanning 24 partition(s)…


,comment_id,docket_id,posted_date,preview
0,EPA-HQ-OAR-2021-0317-2888,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,I am deeply concerned about the impacts of air...
1,EPA-HQ-OAR-2021-0317-2887,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear EPA Administrator Regan,<br/><br/>Methane..."
2,EPA-HQ-OAR-2021-0317-2884,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear Environmental Protection Agency,<br/><br/..."
3,EPA-HQ-OAR-2021-0317-2889,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear Administrator Regan, <br/><br/>Thank you ..."
4,EPA-HQ-OAR-2021-0317-2890,EPA-HQ-OAR-2021-0317,2023-04-07T04:00:00Z,"Dear Administrator Regan, <br/><br/>Thank you ..."


In [17]:
# Agency + year scope — still usually fine, but verify the partition count
# shown by the "scanning N partition(s)…" line stays under MAX_PARTITIONS.
search_comments("forever chemicals", agency="EPA", year=2024, limit=5)

scanning 1137 partition(s)…


,comment_id,docket_id,posted_date,preview
0,EPA-HQ-OPPT-2024-0131-0055,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,"Sunday, December 29, 2024<br/><br/>Office of P..."
1,EPA-HQ-OPPT-2024-0131-0050,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,The science is clear&mdash;PFAS (per and poly-...
2,EPA-HQ-OPPT-2024-0131-0039,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,We must protect the public from &quot;forever ...
3,EPA-HQ-OPPT-2024-0131-0046,EPA-HQ-OPPT-2024-0131,2024-12-31T05:00:00Z,I strongly support the EPA&rsquo;s proposed re...
4,EPA-HQ-OPPT-2023-0538-0535,EPA-HQ-OPPT-2023-0538,2024-12-11T05:00:00Z,I strongly support the EPA&#39;s proposal to a...


### Tips

- **Always scope by `agency`, and ideally also `docket_id` or `year`.** DuckDB issues one HTTP range request per partition file — broad agency-wide searches fan out to hundreds of requests and trip r2.dev's 429 rate limiter. The helper caps at 50 partitions by default.
- **Hit 429 anyway?** Re-run the cell after a minute; the rate limit is short. For sustained use, put the bucket behind a custom domain or proxy.
- **`ILIKE` is substring, not token-aware.** For phrase matching with stemming/fuzziness, pull the matched partition into memory and run a real search lib (Whoosh, rank_bm25) locally.
- **Want ranking?** Add `regexp_matches(comment, ...)` or score by `len(regexp_extract_all(...))` — cheap and runs pushed-down inside DuckDB.

## Bonus — the hosted UI

If you don't want to write code at all, the live site has a search box at **`/search`** backed by exactly the Layer 1 index above (via [MiniSearch](https://github.com/lucaong/minisearch) for fuzzy/prefix ranking). See `frontend/src/lib/search/index.ts` for the client implementation.

## Which layer to use?

- **I want the same search the site has** → Layer 1 (or just use the site).
- **I want SQL flexibility on docket metadata** → Layer 2.
- **I want to search inside comments (the mirrulations-search use case)** → Layer 3, always scoped by agency.

None of the above require running a server.